In [1]:
%%writefile bookings.csv
booking_id,customer_name,city,service_type,provider,booking_amount,booking_status,payment_mode
1001,Aarav Mehta,Hyderabad,Flight,IndiGo,6500,Confirmed,UPI
1002,Sana Khan,Bangalore,Hotel,Pearl Grand,4500,Confirmed,Card
1003,John Mathew,,Flight,Air India,12000,Confirmed,UPI
1004,Ayesha Begum,Hyderabad,Hotel,,7500,Pending,Cash
1005,Vikram Rao,Mumbai,Flight,Vistara,,Confirmed,Card
1006,Divya Sharma,Delhi,Flight,IndiGo,5900,Cancelled,
1007,Imran Ali,Pune,Hotel,Budget Inn,2200,,UPI
1008,Meera Nair,Kochi,Hotel,Hill View Resort,7500,Confirmed,Card
1009,Rohan Das,Kolkata,Flight,Air India,7400,Pending,UPI
1010,Nisha Reddy,Bangalore,Flight,British Airways,62000,Confirmed,Card
1011,Farhan Ali,,Hotel,Skyline Suites,22000,Confirmed,
1012,Neha Singh,Hyderabad,,Emirates,28000,Confirmed,UPI
1013,Arjun Verma,Chennai,Flight,,15000,Cancelled,Cash
1014,Kavya Nair,Mumbai,Hotel,Sea View Stay,,Pending,Card
1015,Ravi Kumar,Delhi,Flight,SpiceJet,4800,Confirmed,UPI

Writing bookings.csv


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, isnan, isnull, avg, sum as spark_sum

spark = SparkSession.builder.appName("NullHandling").getOrCreate()

ModuleNotFoundError: No module named 'pyspark'

## CSV Dataset - bookings.csv

In [ ]:
# 1. Read bookings.csv
df = spark.read.option("header", "true").option("inferSchema", "true").csv("bookings.csv")
df.show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |
|1003      |John Mathew  |NULL     |Flight      |Air India       |12000         |Confirmed     |UPI         |
|1004      |Ayesha Begum |Hyderabad|Hotel       |NULL            |7500          |Pending       |Cash        |
|1005      |Vikram Rao   |Mumbai   |Flight      |Vistara         |NULL          |Confirmed     |Card        |
|1006      |Divya Sharma |Delhi    |Flight      |IndiGo          |5900          |Cancelled     |NULL        |
|1007     

In [ ]:
# 2. Display schema
df.printSchema()

root
 |-- booking_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- service_type: string (nullable = true)
 |-- provider: string (nullable = true)
 |-- booking_amount: integer (nullable = true)
 |-- booking_status: string (nullable = true)
 |-- payment_mode: string (nullable = true)



In [ ]:
# 3. Count total records
print("Total records:", df.count())

Total records: 15


In [ ]:
# 4. Find records where city is null
df.filter(col("city").isNull()).show(truncate=False)

+----------+-------------+----+------------+--------------+--------------+--------------+------------+
|booking_id|customer_name|city|service_type|provider      |booking_amount|booking_status|payment_mode|
+----------+-------------+----+------------+--------------+--------------+--------------+------------+
|1003      |John Mathew  |NULL|Flight      |Air India     |12000         |Confirmed     |UPI         |
|1011      |Farhan Ali   |NULL|Hotel       |Skyline Suites|22000         |Confirmed     |NULL        |
+----------+-------------+----+------------+--------------+--------------+--------------+------------+



In [ ]:
# 5. Find records where provider is null
df.filter(col("provider").isNull()).show(truncate=False)

+----------+-------------+---------+------------+--------+--------------+--------------+------------+
|booking_id|customer_name|city     |service_type|provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+--------+--------------+--------------+------------+
|1004      |Ayesha Begum |Hyderabad|Hotel       |NULL    |7500          |Pending       |Cash        |
|1013      |Arjun Verma  |Chennai  |Flight      |NULL    |15000         |Cancelled     |Cash        |
+----------+-------------+---------+------------+--------+--------------+--------------+------------+



In [ ]:
# 6. Find records where booking_amount is null
df.filter(col("booking_amount").isNull()).show(truncate=False)

+----------+-------------+------+------------+-------------+--------------+--------------+------------+
|booking_id|customer_name|city  |service_type|provider     |booking_amount|booking_status|payment_mode|
+----------+-------------+------+------------+-------------+--------------+--------------+------------+
|1005      |Vikram Rao   |Mumbai|Flight      |Vistara      |NULL          |Confirmed     |Card        |
|1014      |Kavya Nair   |Mumbai|Hotel       |Sea View Stay|NULL          |Pending       |Card        |
+----------+-------------+------+------------+-------------+--------------+--------------+------------+



In [ ]:
# 7. Find records where booking_status is null
df.filter(col("booking_status").isNull()).show(truncate=False)

+----------+-------------+----+------------+----------+--------------+--------------+------------+
|booking_id|customer_name|city|service_type|provider  |booking_amount|booking_status|payment_mode|
+----------+-------------+----+------------+----------+--------------+--------------+------------+
|1007      |Imran Ali    |Pune|Hotel       |Budget Inn|2200          |NULL          |UPI         |
+----------+-------------+----+------------+----------+--------------+--------------+------------+



In [ ]:
# 8. Find records where payment_mode is null
df.filter(col("payment_mode").isNull()).show(truncate=False)

+----------+-------------+-----+------------+--------------+--------------+--------------+------------+
|booking_id|customer_name|city |service_type|provider      |booking_amount|booking_status|payment_mode|
+----------+-------------+-----+------------+--------------+--------------+--------------+------------+
|1006      |Divya Sharma |Delhi|Flight      |IndiGo        |5900          |Cancelled     |NULL        |
|1011      |Farhan Ali   |NULL |Hotel       |Skyline Suites|22000         |Confirmed     |NULL        |
+----------+-------------+-----+------------+--------------+--------------+--------------+------------+



In [ ]:
# 9. Count null values in each column
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

+----------+-------------+----+------------+--------+--------------+--------------+------------+
|booking_id|customer_name|city|service_type|provider|booking_amount|booking_status|payment_mode|
+----------+-------------+----+------------+--------+--------------+--------------+------------+
|         0|            0|   2|           1|       2|             2|             1|           2|
+----------+-------------+----+------------+--------+--------------+--------------+------------+



In [ ]:
# 10. Drop all rows having any null value
df_no_nulls = df.dropna()
df_no_nulls.show(truncate=False)
print("Records after dropping all nulls:", df_no_nulls.count())

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |
|1008      |Meera Nair   |Kochi    |Hotel       |Hill View Resort|7500          |Confirmed     |Card        |
|1009      |Rohan Das    |Kolkata  |Flight      |Air India       |7400          |Pending       |UPI         |
|1010      |Nisha Reddy  |Bangalore|Flight      |British Airways |62000         |Confirmed     |Card        |
|1015      |Ravi Kumar   |Delhi    |Flight      |SpiceJet        |4800          |Confirmed     |UPI         |
+---------

In [ ]:
# 11. Drop rows where booking_amount is null
df_drop_amount = df.dropna(subset=["booking_amount"])
df_drop_amount.show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |
|1003      |John Mathew  |NULL     |Flight      |Air India       |12000         |Confirmed     |UPI         |
|1004      |Ayesha Begum |Hyderabad|Hotel       |NULL            |7500          |Pending       |Cash        |
|1006      |Divya Sharma |Delhi    |Flight      |IndiGo          |5900          |Cancelled     |NULL        |
|1007      |Imran Ali    |Pune     |Hotel       |Budget Inn      |2200          |NULL          |UPI         |
|1008     

In [ ]:
# 12. Drop rows where customer_name, service_type or booking_amount is null
df_drop_multi = df.dropna(subset=["customer_name", "service_type", "booking_amount"])
df_drop_multi.show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |
|1003      |John Mathew  |NULL     |Flight      |Air India       |12000         |Confirmed     |UPI         |
|1004      |Ayesha Begum |Hyderabad|Hotel       |NULL            |7500          |Pending       |Cash        |
|1006      |Divya Sharma |Delhi    |Flight      |IndiGo          |5900          |Cancelled     |NULL        |
|1007      |Imran Ali    |Pune     |Hotel       |Budget Inn      |2200          |NULL          |UPI         |
|1008     

In [ ]:
# 13. Fill null city with Unknown
df_filled = df.fillna({"city": "Unknown"})
df_filled.show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |
|1003      |John Mathew  |Unknown  |Flight      |Air India       |12000         |Confirmed     |UPI         |
|1004      |Ayesha Begum |Hyderabad|Hotel       |NULL            |7500          |Pending       |Cash        |
|1005      |Vikram Rao   |Mumbai   |Flight      |Vistara         |NULL          |Confirmed     |Card        |
|1006      |Divya Sharma |Delhi    |Flight      |IndiGo          |5900          |Cancelled     |NULL        |
|1007     

In [ ]:
# 14. Fill null provider with Not Available
df_filled = df_filled.fillna({"provider": "Not Available"})
df_filled.show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |
|1003      |John Mathew  |Unknown  |Flight      |Air India       |12000         |Confirmed     |UPI         |
|1004      |Ayesha Begum |Hyderabad|Hotel       |Not Available   |7500          |Pending       |Cash        |
|1005      |Vikram Rao   |Mumbai   |Flight      |Vistara         |NULL          |Confirmed     |Card        |
|1006      |Divya Sharma |Delhi    |Flight      |IndiGo          |5900          |Cancelled     |NULL        |
|1007     

In [ ]:
# 15. Fill null payment_mode with Not Provided
df_filled = df_filled.fillna({"payment_mode": "Not Provided"})
df_filled.show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |
|1003      |John Mathew  |Unknown  |Flight      |Air India       |12000         |Confirmed     |UPI         |
|1004      |Ayesha Begum |Hyderabad|Hotel       |Not Available   |7500          |Pending       |Cash        |
|1005      |Vikram Rao   |Mumbai   |Flight      |Vistara         |NULL          |Confirmed     |Card        |
|1006      |Divya Sharma |Delhi    |Flight      |IndiGo          |5900          |Cancelled     |Not Provided|
|1007     

In [ ]:
# 16. Fill null booking_status with Unknown
df_filled = df_filled.fillna({"booking_status": "Unknown"})
df_filled.show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |
|1003      |John Mathew  |Unknown  |Flight      |Air India       |12000         |Confirmed     |UPI         |
|1004      |Ayesha Begum |Hyderabad|Hotel       |Not Available   |7500          |Pending       |Cash        |
|1005      |Vikram Rao   |Mumbai   |Flight      |Vistara         |NULL          |Confirmed     |Card        |
|1006      |Divya Sharma |Delhi    |Flight      |IndiGo          |5900          |Cancelled     |Not Provided|
|1007     

In [ ]:
# 17. Fill null booking_amount with 0
df_filled = df_filled.fillna({"booking_amount": 0})
df_filled.show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |
|1003      |John Mathew  |Unknown  |Flight      |Air India       |12000         |Confirmed     |UPI         |
|1004      |Ayesha Begum |Hyderabad|Hotel       |Not Available   |7500          |Pending       |Cash        |
|1005      |Vikram Rao   |Mumbai   |Flight      |Vistara         |0             |Confirmed     |Card        |
|1006      |Divya Sharma |Delhi    |Flight      |IndiGo          |5900          |Cancelled     |Not Provided|
|1007     

In [ ]:
# 18. Create data_quality_status column
df_quality = df.withColumn(
    "data_quality_status",
    when(
        col("city").isNull() |
        col("provider").isNull() |
        col("booking_amount").isNull() |
        col("booking_status").isNull() |
        col("payment_mode").isNull() |
        col("service_type").isNull(),
        "Incomplete"
    ).otherwise("Complete")
)
df_quality.show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|data_quality_status|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |Complete           |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |Complete           |
|1003      |John Mathew  |NULL     |Flight      |Air India       |12000         |Confirmed     |UPI         |Incomplete         |
|1004      |Ayesha Begum |Hyderabad|Hotel       |NULL            |7500          |Pending       |Cash        |Incomplete         |
|1005      |Vikram Rao   |Mumbai   |Flight      |Vistara         |NULL          |Confirmed

In [ ]:
# 19. Count records by data_quality_status
df_quality.groupBy("data_quality_status").count().show()

+-------------------+-----+
|data_quality_status|count|
+-------------------+-----+
|           Complete|    6|
|         Incomplete|    9|
+-------------------+-----+



In [ ]:
# 20. Display only Complete records
df_quality.filter(col("data_quality_status") == "Complete").show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|data_quality_status|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |Complete           |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |Complete           |
|1008      |Meera Nair   |Kochi    |Hotel       |Hill View Resort|7500          |Confirmed     |Card        |Complete           |
|1009      |Rohan Das    |Kolkata  |Flight      |Air India       |7400          |Pending       |UPI         |Complete           |
|1010      |Nisha Reddy  |Bangalore|Flight      |British Airways |62000         |Confirmed

In [ ]:
# 21. Display only Incomplete records
df_quality.filter(col("data_quality_status") == "Incomplete").show(truncate=False)

+----------+-------------+---------+------------+--------------+--------------+--------------+------------+-------------------+
|booking_id|customer_name|city     |service_type|provider      |booking_amount|booking_status|payment_mode|data_quality_status|
+----------+-------------+---------+------------+--------------+--------------+--------------+------------+-------------------+
|1003      |John Mathew  |NULL     |Flight      |Air India     |12000         |Confirmed     |UPI         |Incomplete         |
|1004      |Ayesha Begum |Hyderabad|Hotel       |NULL          |7500          |Pending       |Cash        |Incomplete         |
|1005      |Vikram Rao   |Mumbai   |Flight      |Vistara       |NULL          |Confirmed     |Card        |Incomplete         |
|1006      |Divya Sharma |Delhi    |Flight      |IndiGo        |5900          |Cancelled     |NULL        |Incomplete         |
|1007      |Imran Ali    |Pune     |Hotel       |Budget Inn    |2200          |NULL          |UPI       

In [ ]:
# 22. Create tax column = booking_amount * 5%
df_tax = df_filled.withColumn("tax", col("booking_amount") * 0.05)
df_tax.show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|tax   |
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |325.0 |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |225.0 |
|1003      |John Mathew  |Unknown  |Flight      |Air India       |12000         |Confirmed     |UPI         |600.0 |
|1004      |Ayesha Begum |Hyderabad|Hotel       |Not Available   |7500          |Pending       |Cash        |375.0 |
|1005      |Vikram Rao   |Mumbai   |Flight      |Vistara         |0             |Confirmed     |Card        |0.0   |
|1006      |Divya Sharma |Delhi    |Flight      |IndiGo         

In [ ]:
# 23. Create final_amount = booking_amount + tax
df_final = df_tax.withColumn("final_amount", col("booking_amount") + col("tax"))
df_final.show(truncate=False)

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+------+------------+
|booking_id|customer_name|city     |service_type|provider        |booking_amount|booking_status|payment_mode|tax   |final_amount|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+------+------------+
|1001      |Aarav Mehta  |Hyderabad|Flight      |IndiGo          |6500          |Confirmed     |UPI         |325.0 |6825.0      |
|1002      |Sana Khan    |Bangalore|Hotel       |Pearl Grand     |4500          |Confirmed     |Card        |225.0 |4725.0      |
|1003      |John Mathew  |Unknown  |Flight      |Air India       |12000         |Confirmed     |UPI         |600.0 |12600.0     |
|1004      |Ayesha Begum |Hyderabad|Hotel       |Not Available   |7500          |Pending       |Cash        |375.0 |7875.0      |
|1005      |Vikram Rao   |Mumbai   |Flight      |Vistara         |0             |Confirmed

In [ ]:
# 24. Calculate revenue only from confirmed bookings
confirmed_revenue = df_filled.filter(col("booking_status") == "Confirmed") \
    .agg(spark_sum("booking_amount").alias("total_confirmed_revenue"))
confirmed_revenue.show()

+-----------------------+
|total_confirmed_revenue|
+-----------------------+
|                 147300|
+-----------------------+



In [ ]:
# 25. Count bookings by service_type after handling nulls
df_filled.fillna({"service_type": "Unknown"}).groupBy("service_type").count().show()

+------------+-----+
|service_type|count|
+------------+-----+
|     Unknown|    1|
|       Hotel|    6|
|      Flight|    8|
+------------+-----+



In [ ]:
# 26. Count bookings by city after filling missing cities
df_filled.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|    Kochi|    1|
|  Chennai|    1|
|   Mumbai|    2|
|  Kolkata|    1|
|  Unknown|    2|
|     Pune|    1|
|    Delhi|    2|
|Hyderabad|    3|
+---------+-----+



In [ ]:
# 27. Find average booking amount after replacing null amount with 0
avg_with_zero = df.fillna({"booking_amount": 0}).agg(avg("booking_amount").alias("avg_with_zero"))
avg_with_zero.show()

+------------------+
|     avg_with_zero|
+------------------+
|12353.333333333334|
+------------------+



In [ ]:
# 28. Find average booking amount after dropping null amount rows
avg_without_nulls = df.dropna(subset=["booking_amount"]).agg(avg("booking_amount").alias("avg_without_nulls"))
avg_without_nulls.show()

+------------------+
| avg_without_nulls|
+------------------+
|14253.846153846154|
+------------------+



In [ ]:
# 29. Compare both averages
avg_with_zero.crossJoin(avg_without_nulls).show()

+------------------+------------------+
|     avg_with_zero| avg_without_nulls|
+------------------+------------------+
|12353.333333333334|14253.846153846154|
+------------------+------------------+



In [ ]:
# 30. Save clean data as clean_bookings.parquet
df_filled.write.mode("overwrite").parquet("clean_bookings.parquet")
print("Saved clean_bookings.parquet")

Saved clean_bookings.parquet


## JSON Dataset - customers.json

In [ ]:
%%writefile customers.json
[
{"customer_id": 1, "name": "Aarav Mehta", "city": "Hyderabad", "membership": "Gold", "contact": {"phone": "9876500011", "email": "aarav@mail.com"}, "preferences": {"preferred_service": "Flight", "budget_range": "Medium"}},
{"customer_id": 2, "name": "Sana Khan", "city": "Bangalore", "membership": "Silver", "contact": {"phone": null, "email": "sana@mail.com"}, "preferences": {"preferred_service": "Hotel", "budget_range": null}},
{"customer_id": 3, "name": "John Mathew", "city": null, "membership": "Gold", "contact": {"phone": "9876500013", "email": null}, "preferences": {"preferred_service": "Flight", "budget_range": "High"}},
{"customer_id": 4, "name": "Ayesha Begum", "city": "Hyderabad", "membership": null, "contact": {"phone": "9876500014", "email": "ayesha@mail.com"}, "preferences": {"preferred_service": null, "budget_range": "Low"}},
{"customer_id": 5, "name": "Vikram Rao", "city": "Mumbai", "membership": "Platinum", "contact": {"phone": null, "email": null}, "preferences": {"preferred_service": "Flight", "budget_range": "High"}}
]

Writing customers.json


In [ ]:
# 1. Read customers.json
customers_df = spark.read.option("multiline", "true").json("customers.json")

In [ ]:
# 2. Display schema
customers_df.printSchema()

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- membership: string (nullable = true)
 |-- name: string (nullable = true)
 |-- preferences: struct (nullable = true)
 |    |-- budget_range: string (nullable = true)
 |    |-- preferred_service: string (nullable = true)



In [ ]:
# 3. Display all customer records
customers_df.show(truncate=False)

+---------+-----------------------------+-----------+----------+------------+----------------+
|city     |contact                      |customer_id|membership|name        |preferences     |
+---------+-----------------------------+-----------+----------+------------+----------------+
|Hyderabad|{aarav@mail.com, 9876500011} |1          |Gold      |Aarav Mehta |{Medium, Flight}|
|Bangalore|{sana@mail.com, NULL}        |2          |Silver    |Sana Khan   |{NULL, Hotel}   |
|NULL     |{NULL, 9876500013}           |3          |Gold      |John Mathew |{High, Flight}  |
|Hyderabad|{ayesha@mail.com, 9876500014}|4          |NULL      |Ayesha Begum|{Low, NULL}     |
|Mumbai   |{NULL, NULL}                 |5          |Platinum  |Vikram Rao  |{High, Flight}  |
+---------+-----------------------------+-----------+----------+------------+----------------+



In [ ]:
# 4 & 5. Flatten all nested fields
flat_customers_df = customers_df.select(
    "customer_id",
    "name",
    "city",
    "membership",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email"),
    col("preferences.preferred_service").alias("preferred_service"),
    col("preferences.budget_range").alias("budget_range")
)
flat_customers_df.show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [ ]:
# 6. Display customer name, city, phone and email
flat_customers_df.select("name", "city", "phone", "email").show(truncate=False)

+------------+---------+----------+---------------+
|name        |city     |phone     |email          |
+------------+---------+----------+---------------+
|Aarav Mehta |Hyderabad|9876500011|aarav@mail.com |
|Sana Khan   |Bangalore|NULL      |sana@mail.com  |
|John Mathew |NULL     |9876500013|NULL           |
|Ayesha Begum|Hyderabad|9876500014|ayesha@mail.com|
|Vikram Rao  |Mumbai   |NULL      |NULL           |
+------------+---------+----------+---------------+



In [ ]:
# 7. Find customers where city is null
flat_customers_df.filter(col("city").isNull()).show(truncate=False)

+-----------+-----------+----+----------+----------+-----+-----------------+------------+
|customer_id|name       |city|membership|phone     |email|preferred_service|budget_range|
+-----------+-----------+----+----------+----------+-----+-----------------+------------+
|3          |John Mathew|NULL|Gold      |9876500013|NULL |Flight           |High        |
+-----------+-----------+----+----------+----------+-----+-----------------+------------+



In [ ]:
# 8. Find customers where phone is null
flat_customers_df.filter(col("phone").isNull()).show(truncate=False)

+-----------+----------+---------+----------+-----+-------------+-----------------+------------+
|customer_id|name      |city     |membership|phone|email        |preferred_service|budget_range|
+-----------+----------+---------+----------+-----+-------------+-----------------+------------+
|2          |Sana Khan |Bangalore|Silver    |NULL |sana@mail.com|Hotel            |NULL        |
|5          |Vikram Rao|Mumbai   |Platinum  |NULL |NULL         |Flight           |High        |
+-----------+----------+---------+----------+-----+-------------+-----------------+------------+



In [ ]:
# 9. Find customers where email is null
flat_customers_df.filter(col("email").isNull()).show(truncate=False)

+-----------+-----------+------+----------+----------+-----+-----------------+------------+
|customer_id|name       |city  |membership|phone     |email|preferred_service|budget_range|
+-----------+-----------+------+----------+----------+-----+-----------------+------------+
|3          |John Mathew|NULL  |Gold      |9876500013|NULL |Flight           |High        |
|5          |Vikram Rao |Mumbai|Platinum  |NULL      |NULL |Flight           |High        |
+-----------+-----------+------+----------+----------+-----+-----------------+------------+



In [ ]:
# 10. Find customers where membership is null
flat_customers_df.filter(col("membership").isNull()).show(truncate=False)

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|name        |city     |membership|phone     |email          |preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|4          |Ayesha Begum|Hyderabad|NULL      |9876500014|ayesha@mail.com|NULL             |Low         |
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [ ]:
# 11. Find customers where preferred_service is null
flat_customers_df.filter(col("preferred_service").isNull()).show(truncate=False)

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|name        |city     |membership|phone     |email          |preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|4          |Ayesha Begum|Hyderabad|NULL      |9876500014|ayesha@mail.com|NULL             |Low         |
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [ ]:
# 12. Find customers where budget_range is null
flat_customers_df.filter(col("budget_range").isNull()).show(truncate=False)

+-----------+---------+---------+----------+-----+-------------+-----------------+------------+
|customer_id|name     |city     |membership|phone|email        |preferred_service|budget_range|
+-----------+---------+---------+----------+-----+-------------+-----------------+------------+
|2          |Sana Khan|Bangalore|Silver    |NULL |sana@mail.com|Hotel            |NULL        |
+-----------+---------+---------+----------+-----+-------------+-----------------+------------+



In [ ]:
# 13. Count null values in flattened customer DataFrame
flat_customers_df.select([count(when(col(c).isNull(), c)).alias(c) for c in flat_customers_df.columns]).show()

+-----------+----+----+----------+-----+-----+-----------------+------------+
|customer_id|name|city|membership|phone|email|preferred_service|budget_range|
+-----------+----+----+----------+-----+-----+-----------------+------------+
|          0|   0|   1|         1|    2|    2|                1|           1|
+-----------+----+----+----------+-----+-----+-----------------+------------+



In [ ]:
# 14. Fill null city with Unknown
clean_customers = flat_customers_df.fillna({"city": "Unknown"})
clean_customers.show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|  Unknown|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [ ]:
# 15. Fill null membership with Standard
clean_customers = clean_customers.fillna({"membership": "Standard"})
clean_customers.show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|  Unknown|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|  Standard|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [ ]:
# 16. Fill null phone with Not Provided
clean_customers = clean_customers.fillna({"phone": "Not Provided"})
clean_customers.show()

+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|       phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|  9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|Not Provided|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|  Unknown|      Gold|  9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|  Standard|  9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|Not Provided|           NULL|           Flight|        High|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+



In [ ]:
# 17. Fill null email with Not Provided
clean_customers = clean_customers.fillna({"email": "Not Provided"})
clean_customers.show()

+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|       phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|  9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|Not Provided|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|  Unknown|      Gold|  9876500013|   Not Provided|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|  Standard|  9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|Not Provided|   Not Provided|           Flight|        High|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+



In [ ]:
# 18. Fill null preferred_service with Not Selected
clean_customers = clean_customers.fillna({"preferred_service": "Not Selected"})
clean_customers.show()

+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|       phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|  9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|Not Provided|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|  Unknown|      Gold|  9876500013|   Not Provided|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|  Standard|  9876500014|ayesha@mail.com|     Not Selected|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|Not Provided|   Not Provided|           Flight|        High|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+



In [ ]:
# 19. Fill null budget_range with Unknown
clean_customers = clean_customers.fillna({"budget_range": "Unknown"})
clean_customers.show()

+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|       phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|  9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|Not Provided|  sana@mail.com|            Hotel|     Unknown|
|          3| John Mathew|  Unknown|      Gold|  9876500013|   Not Provided|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|  Standard|  9876500014|ayesha@mail.com|     Not Selected|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|Not Provided|   Not Provided|           Flight|        High|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+



In [ ]:
# 20. Create customer_quality_status
cust_quality = flat_customers_df.withColumn(
    "customer_quality_status",
    when(
        col("city").isNull() |
        col("phone").isNull() |
        col("email").isNull() |
        col("membership").isNull() |
        col("preferred_service").isNull(),
        "Incomplete"
    ).otherwise("Complete")
)
cust_quality.show(truncate=False)

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+
|customer_id|name        |city     |membership|phone     |email          |preferred_service|budget_range|customer_quality_status|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+
|1          |Aarav Mehta |Hyderabad|Gold      |9876500011|aarav@mail.com |Flight           |Medium      |Complete               |
|2          |Sana Khan   |Bangalore|Silver    |NULL      |sana@mail.com  |Hotel            |NULL        |Incomplete             |
|3          |John Mathew |NULL     |Gold      |9876500013|NULL           |Flight           |High        |Incomplete             |
|4          |Ayesha Begum|Hyderabad|NULL      |9876500014|ayesha@mail.com|NULL             |Low         |Incomplete             |
|5          |Vikram Rao  |Mumbai   |Platinum  |NULL      |NULL           |Flight          

In [ ]:
# 21. Count customers by customer_quality_status
cust_quality.groupBy("customer_quality_status").count().show()

+-----------------------+-----+
|customer_quality_status|count|
+-----------------------+-----+
|               Complete|    1|
|             Incomplete|    4|
+-----------------------+-----+



In [ ]:
# 22. Display only complete customers
cust_quality.filter(col("customer_quality_status") == "Complete").show(truncate=False)

+-----------+-----------+---------+----------+----------+--------------+-----------------+------------+-----------------------+
|customer_id|name       |city     |membership|phone     |email         |preferred_service|budget_range|customer_quality_status|
+-----------+-----------+---------+----------+----------+--------------+-----------------+------------+-----------------------+
|1          |Aarav Mehta|Hyderabad|Gold      |9876500011|aarav@mail.com|Flight           |Medium      |Complete               |
+-----------+-----------+---------+----------+----------+--------------+-----------------+------------+-----------------------+



In [ ]:
# 23. Display only incomplete customers
cust_quality.filter(col("customer_quality_status") == "Incomplete").show(truncate=False)

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+
|customer_id|name        |city     |membership|phone     |email          |preferred_service|budget_range|customer_quality_status|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+
|2          |Sana Khan   |Bangalore|Silver    |NULL      |sana@mail.com  |Hotel            |NULL        |Incomplete             |
|3          |John Mathew |NULL     |Gold      |9876500013|NULL           |Flight           |High        |Incomplete             |
|4          |Ayesha Begum|Hyderabad|NULL      |9876500014|ayesha@mail.com|NULL             |Low         |Incomplete             |
|5          |Vikram Rao  |Mumbai   |Platinum  |NULL      |NULL           |Flight           |High        |Incomplete             |
+-----------+------------+---------+----------+----------+---------------+----------------

In [ ]:
# 24. Count customers by membership after handling nulls
clean_customers.groupBy("membership").count().show()

+----------+-----+
|membership|count|
+----------+-----+
|  Platinum|    1|
|    Silver|    1|
|      Gold|    2|
|  Standard|    1|
+----------+-----+



In [ ]:
# 25. Count customers by preferred_service after handling nulls
clean_customers.groupBy("preferred_service").count().show()

+-----------------+-----+
|preferred_service|count|
+-----------------+-----+
|     Not Selected|    1|
|            Hotel|    1|
|           Flight|    3|
+-----------------+-----+



In [ ]:
# 26. Save flattened customer data as customers_flat.parquet
flat_customers_df.write.mode("overwrite").parquet("customers_flat.parquet")
print("Saved customers_flat.parquet")

Saved customers_flat.parquet


In [ ]:
# 27. Save clean customer data as clean_customers.csv
clean_customers.write.mode("overwrite").option("header", "true").csv("clean_customers.csv")
print("Saved clean_customers.csv")

Saved clean_customers.csv


In [ ]:
# 28. Compare original record count and clean record count
print("Original count:", flat_customers_df.count())
print("Clean count:", clean_customers.count())

Original count: 5
Clean count: 5


In [ ]:
# 29. Display customers with missing contact details
flat_customers_df.filter(col("phone").isNull() | col("email").isNull()).show(truncate=False)

+-----------+-----------+---------+----------+----------+-------------+-----------------+------------+
|customer_id|name       |city     |membership|phone     |email        |preferred_service|budget_range|
+-----------+-----------+---------+----------+----------+-------------+-----------------+------------+
|2          |Sana Khan  |Bangalore|Silver    |NULL      |sana@mail.com|Hotel            |NULL        |
|3          |John Mathew|NULL     |Gold      |9876500013|NULL         |Flight           |High        |
|5          |Vikram Rao |Mumbai   |Platinum  |NULL      |NULL         |Flight           |High        |
+-----------+-----------+---------+----------+----------+-------------+-----------------+------------+



In [ ]:
# 30. Display customers with missing preference details
flat_customers_df.filter(col("preferred_service").isNull() | col("budget_range").isNull()).show(truncate=False)

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|name        |city     |membership|phone     |email          |preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|2          |Sana Khan   |Bangalore|Silver    |NULL      |sana@mail.com  |Hotel            |NULL        |
|4          |Ayesha Begum|Hyderabad|NULL      |9876500014|ayesha@mail.com|NULL             |Low         |
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+

